# LLM-RAG experiment for the DASS-21 hybrid testing framework (Article 2, protocol step 7)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gersonfeoh/dass21-hybrid-ai-testing/blob/main/colab/DASS21_LLM_RAG_Colab.ipynb)

This notebook runs the LLM-RAG component and the integrated configurations of the framework:

* **Table 11 pools**: temperature 0 (one sample per target) and temperature 0.7 (ten samples per target), 12 test targets.
* **B4**: validated LLM-RAG tests only; **B5**: lattice-aware GA with Z3 obligations and the validated LLM tests fixed inside the suite;
  **A1**: B5 without Z3; **A3**: B5 with the raw genotype; **A2** (B5 without LLM seeds) is B3 from the main runs.

The LLM is served locally by **Ollama inside this Colab runtime** (open-weight, pinned tag, digest recorded).
The LLM never provides expected outputs: every proposed test is re-scored by the deterministic reference oracle.
No student data are used; all inputs are synthetic response vectors.

**Runtime**: choose *Runtime > Change runtime type > T4 GPU*. A full run (30 seeds) takes about 1 to 1.5 hours;
set `QUICK = True` for a 10-minute check. The run resumes from `results_llm/llm_log.jsonl` if the session reconnects.

**Subject S2** (code-level mutants of the platform) needs the private bundle `s2_private.zip`, which is not public
because it contains production source code. Without it, all S1 measurements and the LLM metrics are still produced.

Repository: https://github.com/gersonfeoh/dass21-hybrid-ai-testing

In [ ]:
# Settings
MODEL = 'qwen2.5:7b-instruct'      # open-weight LLM served by Ollama (tag pinned, digest recorded below)
EMBED = 'bge-m3'                   # open multilingual embedding model for dense retrieval
QUICK = False                      # True: 2 seeds, 2 pool samples, GA budget 2,000 (smoke test only)
SEEDS, POOL, BUDGET = (2, 2, 2000) if QUICK else (30, 10, 20000)
REPO_URL = 'https://github.com/gersonfeoh/dass21-hybrid-ai-testing.git'

In [ ]:
# 1. Code and Python dependencies
import os, sys, subprocess
if not os.path.exists('/content/dass21-hybrid-ai-testing'):
    subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/dass21-hybrid-ai-testing'], check=True)
os.chdir('/content/dass21-hybrid-ai-testing')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print(subprocess.run(['git', 'log', '-1', '--format=%H %cd'], capture_output=True, text=True).stdout)

In [ ]:
# 2. Ollama server inside the Colab runtime
import shutil, time, urllib.request
if shutil.which('ollama') is None:
    subprocess.run('apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null', shell=True, check=True)
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
def ollama_up():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=5); return True
    except Exception:
        return False
if not ollama_up():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(120):
        if ollama_up(): break
        time.sleep(1)
print('Ollama running:', ollama_up())
print(subprocess.run(['ollama', '--version'], capture_output=True, text=True).stdout)
print(subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 3. Models (pinned tags; the digests are written to results_llm/llm_metrics.json)
for m in (MODEL, EMBED):
    subprocess.run(['ollama', 'pull', m], check=True)
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

In [ ]:
# 4. Optional: private S2 bundle (upload s2_private.zip to /content with the Files panel before running this cell)
import zipfile, hashlib
EXPECTED = '4d9b4f6ca78903d2fe0fe40c72e42a5c7cf4716bb39382366991d5f99f58c5d4'
if os.path.exists('/content/s2_private.zip'):
    zipfile.ZipFile('/content/s2_private.zip').extractall('.')
    h = hashlib.sha256(open('s2/src/dass21.js', 'rb').read()).hexdigest()
    assert h == EXPECTED, 'unexpected S2 module'
    print('S2 bundle installed; node', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout)
else:
    print('No private S2 bundle: S2 measurements are skipped, everything else runs.')

In [ ]:
# 5. Run the experiment (generation, validation, GA runs B5/A1/A3, measurement, statistics)
cmd = [sys.executable, 'llm_rag/llm_experiment.py', '--model', MODEL, '--embed-model', EMBED,
       '--seeds', str(SEEDS), '--pool-samples', str(POOL), '--budget', str(BUDGET), '--workers', '2', '--out', 'results_llm']
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if line.startswith(('generated', 'GA runs', 'Traceback')) or 'Error' in line:
        print(line, end='')
p.wait(); print('exit code', p.returncode)

In [ ]:
# 6. Results
import json, pandas as pd
m = json.load(open('results_llm/llm_metrics.json'))
print('model:', m['meta']['model'], '| embedding:', m['meta']['embed_model'], '| S2 included:', m['meta']['s2_included'])
display(pd.DataFrame(m['metrics']).T.drop(columns=['reject_reasons', 'novel_kill_pools'], errors='ignore'))
s = json.load(open('results_llm/stats_llm.json'))
rows = {c: {k: (v['median'] if isinstance(v, dict) else v) for k, v in d.items()} for c, d in s['descriptive'].items()}
display(pd.DataFrame(rows).T)
for metric, tests in s['mann_whitney'].items():
    display(pd.DataFrame(tests).assign(metric=metric))

In [ ]:
# 7. Package the results (logs, metrics, runs) for the manuscript
shutil.make_archive('/content/results_llm', 'zip', 'results_llm')
print('/content/results_llm.zip', os.path.getsize('/content/results_llm.zip'), 'bytes')
try:
    from google.colab import files
    files.download('/content/results_llm.zip')
except Exception as e:
    print('download manually from the Files panel:', e)